# 02 Electrostatic field and capacitance

Uses `scikit-fem` to extract the 4x4 capacitance matrix $C'_{node}$ (F/m), total capacitance $C_{node}$ (F), differential capacitance $C_{diff}$ (2x2), and the Gaussian-beam-weighted field-per-volt matrix $A$.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from rot_eo_model.constants import DEFAULTS
from rot_eo_model.tensors import CapacitanceMatrix, FieldPerVoltMatrix
from rot_eo_model.electrostatics import (
    build_parallel_plate_mesh, 
    build_four_electrode_mesh, 
    ElectrostaticModel2D,
    extract_field_per_volt_matrix,
    EPSILON_0
)

## Part 1: Parallel-plate benchmark test
Validate the FEM charge integration by computing capacitance for an ideal parallel plate.

In [ ]:
width = 10e-3
gap = 1e-3
eps_r = 44.0

mesh_pp = build_parallel_plate_mesh(width, gap, res=30)
model_pp = ElectrostaticModel2D(mesh_pp, lambda p: np.full(p.shape[1], eps_r))

C_node_prime = model_pp.extract_capacitance_matrix_per_length(['top', 'bottom'])
C_analytical = EPSILON_0 * eps_r * width / gap

print(f"FEM C' = {C_node_prime[0,0]*1e12:.3f} pF/m")
print(f"Analytical C' = {C_analytical*1e12:.3f} pF/m")
print(f"Error = {abs(C_node_prime[0,0] - C_analytical)/C_analytical * 100:.2f}%")

## Part 2: Four-electrode square-cross-section
Simulate the actual $3 \times 3$ mm cross-section with four electrodes.

In [ ]:
crystal_size = 3e-3
electrode_width = 1e-3
eps_perp = 44.0 # For LiNbO3 z-cut cross-section

mesh_4el = build_four_electrode_mesh(crystal_size, crystal_size, electrode_width, res=50)
model_4el = ElectrostaticModel2D(mesh_4el, lambda p: np.full(p.shape[1], eps_perp))

electrode_names = ['+x', '-x', '+y', '-y']


## Part 3 & 4: Maxwell Capacitance Matrix & Differential Transformation

In [ ]:
# 1. Compute C'_node (F/m)
C_node_prime = model_4el.extract_capacitance_matrix_per_length(electrode_names)

# 2. Convert to total C_node (F)
C_node = C_node_prime * DEFAULTS.crystal_length_m

# 3. Check symmetry and energy
T = 0.5 * np.array([
    [ 1,  0],
    [-1,  0],
    [ 0,  1],
    [ 0, -1]
])
cap_matrix = CapacitanceMatrix(C_node=C_node, T=T)
print(f"C_node is symmetric: {cap_matrix.verify_symmetry()}")

# 4. Extract C_diff
C_diff = cap_matrix.C_diff

print("\nC_diff (pF):")
print(np.round(C_diff * 1e12, 3))
print(f"\nWorking assumption was 13 pF. Actual C_diff_xx is {C_diff[0,0]*1e12:.2f} pF")

## Part 5 & 6: Field-per-volt Matrix A and Maps

In [ ]:
# Generate grid for points
x = np.linspace(-crystal_size/2, crystal_size/2, 50)
y = np.linspace(-crystal_size/2, crystal_size/2, 50)
X, Y = np.meshgrid(x, y)
points = np.vstack((X.flatten(), Y.flatten()))

w0 = 0.5e-3 # 500 um beam waist
field_metrics = extract_field_per_volt_matrix(model_4el, electrode_names, points, w0)

print("Field-per-volt matrix A (V/m per V_diff):")
print(np.round(field_metrics.A, 1))
print("\nRMS Nonuniformity (V/m per V_diff):")
print(np.round(field_metrics.rms_nonuniformity, 1))

In [ ]:
# Plot potential and field for an X-drive
bc_x = {'+x': 0.5, '-x': -0.5, '+y': 0.0, '-y': 0.0}
ux = model_4el.solve_potential(bc_x)
Ex_vec = model_4el.extract_electric_field(ux, points)

fig, ax = plt.subplots(figsize=(6,5))
c = ax.tricontourf(mesh_4el.p[0]*1e3, mesh_4el.p[1]*1e3, ux, levels=20, cmap='RdBu_r')
fig.colorbar(c, ax=ax, label='Potential (V) [For 1V diff]')

# Overlay field vectors (subsampled for clarity)
step = 3
ax.quiver(points[0, ::step]*1e3, points[1, ::step]*1e3, 
          Ex_vec[0, ::step], Ex_vec[1, ::step], 
          color='k', alpha=0.5, scale=5000)

ax.set_title("X-drive Potential and Electric Field")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
plt.axis('equal')
plt.tight_layout()
plt.show()